# Baselines com Scikit-Learn

## Objetivo

Construir uma referência reproduzível com a API do Scikit-Learn antes de avançar para modelos mais complexos. Serão comparados três níveis:

1. popularidade, que ordena filmes sem prever ratings;
2. média global com `DummyRegressor`, que valida o fluxo mínimo `fit → predict`;
3. vieses aditivos de usuário e filme com `OneHotEncoder + SGDRegressor`.

A escolha de hiperparâmetros usa somente validação. O teste continua fechado até o modelo final ser escolhido.

In [1]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError("Não foi possível localizar data/raw.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from data.loaders import load_movies, load_ratings
from data.splitting import temporal_leave_two_out
from evaluation.metrics import evaluate_top_k, mae, rmse
from models.factory import create_model

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

TOP_K = 10
RELEVANCE_THRESHOLD = 4.0
RANDOM_STATE = 42

## 1. Dados e separação temporal

Para cada usuário, a última interação vai para teste, a penúltima para validação e as anteriores para treino. Assim, o modelo aprende com o passado e é avaliado em eventos futuros, evitando vazamento temporal.

In [2]:
raw_data_dir = PROJECT_ROOT / "data" / "raw"
ratings = load_ratings(raw_data_dir)
movies = load_movies(raw_data_dir)
train, validation, test = temporal_leave_two_out(ratings)

pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [len(train), len(validation), len(test)],
        "users": [
            train["user_id"].nunique(),
            validation["user_id"].nunique(),
            test["user_id"].nunique(),
        ],
        "movies": [
            train["movie_id"].nunique(),
            validation["movie_id"].nunique(),
            test["movie_id"].nunique(),
        ],
    }
)

,split,rows,users,movies
0,train,99616,610,9681
1,validation,610,610,512
2,test,610,610,514


## 2. O que cada baseline aprende

O `DummyRegressor` prevê a média dos ratings do treino para qualquer par. Ele não personaliza nem diferencia filmes; sua função é responder se um modelo treinável supera a regra mais simples possível.

No baseline de vieses, usuário e filme são identificadores categóricos. O `OneHotEncoder` converte cada ID em uma coluna binária esparsa e o `SGDRegressor` aprende um intercepto, um peso por usuário e um peso por filme:

`rating previsto = intercepto + viés do usuário + viés do filme`

Esse modelo melhora a calibração dos ratings, mas ainda não aprende combinações específicas usuário-filme. Para um mesmo usuário, seu termo é constante; portanto, a ordenação tende a ser determinada principalmente pelo viés dos filmes.

In [3]:
def evaluate_rating_model(name, model, fit_data, holdout):
    fitted_model = model.fit(fit_data)
    predictions = fitted_model.predict_pairs(holdout)
    ranking, _, _ = evaluate_top_k(
        holdout,
        fitted_model.recommend,
        fitted_model.catalog_size,
        k=TOP_K,
        relevance_threshold=RELEVANCE_THRESHOLD,
    )
    actual = holdout["rating"].to_numpy(float)
    metrics = {
        "model": name,
        "rmse": rmse(actual, predictions),
        "mae": mae(actual, predictions),
        "precision@10": ranking["precision@10"],
        "recall@10": ranking["recall@10"],
        "ndcg@10": ranking["ndcg@10"],
        "hit_rate@10": ranking["hit_rate@10"],
        "coverage@10": ranking["catalog_coverage@10"],
    }
    return fitted_model, metrics

## 3. Seleção na validação

`alpha` controla a regularização L2: valores maiores encolhem mais os pesos e reduzem o risco de memorizar usuários ou filmes com poucas avaliações. Todos os candidatos veem exatamente o mesmo treino e a mesma validação.

In [4]:
candidate_specs = {
    "sklearn_mean": ("sklearn_mean", {}),
    "sklearn_bias_alpha_1e-4": ("sklearn_bias", {"alpha": 1e-4}),
    "sklearn_bias_alpha_1e-3": ("sklearn_bias", {"alpha": 1e-3}),
    "sklearn_bias_alpha_1e-2": ("sklearn_bias", {"alpha": 1e-2}),
}

validation_rows = []
validation_models = {}
for name, (kind, parameters) in candidate_specs.items():
    model = create_model(kind, random_state=RANDOM_STATE, **parameters) if parameters else create_model(kind)
    fitted, metrics = evaluate_rating_model(name, model, train, validation)
    validation_models[name] = fitted
    validation_rows.append(metrics)

validation_results = pd.DataFrame(validation_rows).sort_values(
    ["ndcg@10", "rmse"], ascending=[False, True]
)
validation_results

,model,rmse,mae,precision@10,recall@10,ndcg@10,hit_rate@10,coverage@10
1,sklearn_bias_alpha_1e-4,0.9489,0.7595,0.0072,0.0720,0.0363,0.0720,0.0093
2,sklearn_bias_alpha_1e-3,1.0579,0.8637,0.0072,0.0720,0.0349,0.0720,0.0104
3,sklearn_bias_alpha_1e-2,1.1063,0.9241,0.0069,0.0692,0.0331,0.0692,0.0105
0,sklearn_mean,1.0753,0.8813,0.0006,0.0058,0.0025,0.0058,0.0031


In [5]:
popularity = create_model("popularity").fit(train)
popularity_ranking, _, _ = evaluate_top_k(
    validation,
    popularity.recommend,
    popularity.catalog_size,
    k=TOP_K,
    relevance_threshold=RELEVANCE_THRESHOLD,
)

pd.DataFrame(
    [
        {
            "model": "popularity",
            "precision@10": popularity_ranking["precision@10"],
            "recall@10": popularity_ranking["recall@10"],
            "ndcg@10": popularity_ranking["ndcg@10"],
            "hit_rate@10": popularity_ranking["hit_rate@10"],
            "coverage@10": popularity_ranking["catalog_coverage@10"],
        }
    ]
).set_index("model")

,precision@10,recall@10,ndcg@10,hit_rate@10,coverage@10
model,,,,,
popularity,0.0043,0.0432,0.0242,0.0432,0.0125


### Como ler as métricas

- RMSE pune erros grandes com mais força; MAE mostra o erro absoluto médio em estrelas.
- Precision@10 mede quantos dos dez itens indicados são relevantes.
- Recall@10 mede quanto dos itens relevantes futuros foi recuperado.
- NDCG@10 também considera a posição: um acerto no topo vale mais.
- Coverage@10 mede a fração do catálogo que apareceu nas listas.

Neste experimento, NDCG@10 é o critério principal porque o produto exibe uma lista ordenada. RMSE desempata configurações com o mesmo ranking.

In [6]:
bias_results = validation_results[
    validation_results["model"].str.startswith("sklearn_bias")
].sort_values(["ndcg@10", "rmse"], ascending=[False, True])
best_name = bias_results.iloc[0]["model"]
best_kind, best_parameters = candidate_specs[best_name]

pd.Series(
    {
        "selected_model": best_name,
        "selection_metric": "ndcg@10 (rmse como desempate)",
        **best_parameters,
    },
    name="selection",
)

selected_model            sklearn_bias_alpha_1e-4
selection_metric    ndcg@10 (rmse como desempate)
alpha                                      0.0001
Name: selection, dtype: object

## 4. Treino final e teste

Depois da escolha, validação deixa de ser necessária para decidir. Ela é anexada ao treino para fornecer mais histórico ao modelo final. Só então medimos uma vez no teste, que simula o futuro ainda não observado.

In [7]:
train_validation = pd.concat([train, validation], ignore_index=True)
final_model = create_model(
    best_kind, random_state=RANDOM_STATE, **best_parameters
)
final_model, test_metrics = evaluate_rating_model(
    best_name, final_model, train_validation, test
)
pd.Series(test_metrics, name="test")

model           sklearn_bias_alpha_1e-4
rmse                             1.0176
mae                              0.8158
precision@10                     0.0061
recall@10                        0.0606
ndcg@10                          0.0311
hit_rate@10                      0.0606
coverage@10                      0.0090
Name: test, dtype: object

## 5. Inspeção das recomendações

Métricas agregadas não mostram se as listas fazem sentido. A inspeção abaixo traduz os IDs para títulos e confirma que itens já presentes no treino final foram filtrados.

In [8]:
sample_users = test["user_id"].drop_duplicates().head(3).tolist()
for user_id in sample_users:
    movie_ids = final_model.recommend(int(user_id), TOP_K)
    ranked = pd.DataFrame(
        {"rank": range(1, len(movie_ids) + 1), "movie_id": movie_ids}
    ).merge(movies, on="movie_id", how="left")
    seen = set(train_validation.loc[train_validation["user_id"] == user_id, "movie_id"])
    assert not set(movie_ids) & seen
    print(f"\nUsuário {user_id}")
    display(ranked[["rank", "title", "genres"]])


Usuário 1


,rank,title,genres
0,1,"Shawshank Redemption, The (1994)",Crime|Drama
1,2,"Godfather, The (1972)",Crime|Drama
2,3,"Dark Knight, The (2008)",Action|Crime|Drama|IMAX
3,4,"Godfather: Part II, The (1974)",Crime|Drama
4,5,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy|War
5,6,One Flew Over the Cuckoo's Nest (1975),Drama
6,7,"Lord of the Rings: The Return of the King, The...",Action|Adventure|Drama|Fantasy
7,8,Eternal Sunshine of the Spotless Mind (2004),Drama|Romance|Sci-Fi
8,9,"Amelie (Fabuleux destin d'Amélie Poulain, Le) ...",Comedy|Romance
9,10,"Departed, The (2006)",Crime|Drama|Thriller



Usuário 2


,rank,title,genres
0,1,Fight Club (1999),Action|Crime|Drama|Thriller
1,2,"Usual Suspects, The (1995)",Crime|Mystery|Thriller
2,3,"Godfather, The (1972)",Crime|Drama
3,4,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi
4,5,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller
5,6,Schindler's List (1993),Drama|War
6,7,Forrest Gump (1994),Comedy|Drama|Romance|War
7,8,Raiders of the Lost Ark (Indiana Jones and the...,Action|Adventure
8,9,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Sci-Fi
9,10,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller



Usuário 3


,rank,title,genres
0,1,"Shawshank Redemption, The (1994)",Crime|Drama
1,2,Fight Club (1999),Action|Crime|Drama|Thriller
2,3,"Usual Suspects, The (1995)",Crime|Mystery|Thriller
3,4,"Godfather, The (1972)",Crime|Drama
4,5,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi
5,6,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller
6,7,Forrest Gump (1994),Comedy|Drama|Romance|War
7,8,Raiders of the Lost Ark (Indiana Jones and the...,Action|Adventure
8,9,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Sci-Fi
9,10,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller


## Conclusões

- O adaptador comum conecta a API do Scikit-Learn ao contrato do projeto: `fit`, `predict_pairs`, `recommend` e `catalog_size`.
- O split temporal conecta a lógica de dados ao treinamento e impede que eventos futuros sejam usados para prever o passado.
- O `DummyRegressor` valida o piso de regressão; o modelo de vieses mede quanto usuário e popularidade média do filme explicam sem interações latentes.
- IDs desconhecidos são tratados pelo `OneHotEncoder(handle_unknown="ignore")`; ratings previstos são limitados ao intervalo de 0,5 a 5,0.
- Como o baseline aditivo não modela afinidades usuário-filme, fatoração de matriz e embeddings continuam sendo os modelos adequados para testar personalização real.